# 01 - Generate Synthetic AML Data

## Goal

Run All generates only synthetic CSV data required by the AML assignment. This notebook does not build features, models, rules, APIs, or dashboards.

## 1. Setup

The notebook locates the workspace and creates the two output folders. Dependencies: `numpy` and `pandas`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "notebooks").exists():
    ROOT = ROOT.parent
if not (ROOT / "notebooks").exists():
    raise RuntimeError("Open this notebook from inside the AML workspace.")

RAW_DIR = ROOT / "data" / "raw"
TRUTH_DIR = ROOT / "data" / "ground_truth"
RAW_DIR.mkdir(parents=True, exist_ok=True)
TRUTH_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")
ROOT


WindowsPath('E:/Trading/V-Teki Project/Anti Money Laundering Detection Updated')

## 2. Parameters and helper data

`full` is the default and follows the PDF minimum volume. `smoke` is available only for a fast local test.

In [2]:
RUN_SCALE = "full"  # Run All creates the PDF-scale data. Change to "smoke" only for a fast check.
SEED = 42

SIZES = {
    "full": {"customers": 10_000, "accounts": 15_000, "counterparties": 5_000, "watchlist": 1_200, "transactions": 250_000, "history_days": 240, "labels_per_scenario": 100, "sanctions_cases": 750},
    "smoke": {"customers": 500, "accounts": 750, "counterparties": 300, "watchlist": 120, "transactions": 6_000, "history_days": 60, "labels_per_scenario": 20, "sanctions_cases": 60},
}
if RUN_SCALE not in SIZES:
    raise ValueError("RUN_SCALE must be 'full' or 'smoke'.")
cfg = SIZES[RUN_SCALE]
rng = np.random.default_rng(SEED)

FIRST = np.array(["Adit", "Alya", "Andi", "Anisa", "Arif", "Bagus", "Bima", "Citra", "Dewi", "Dimas", "Eka", "Farah", "Fajar", "Gita", "Hana", "Indra", "Intan", "Joko", "Kevin", "Laras", "Maya", "Nadia", "Naufal", "Putri", "Raka", "Rani", "Rizky", "Sari", "Teguh", "Wulan"])
LAST = np.array(["Adinata", "Budiman", "Chandra", "Darmawan", "Firmansyah", "Gunawan", "Halim", "Hartono", "Iskandar", "Kurniawan", "Lesmana", "Mahendra", "Nugraha", "Permana", "Prakoso", "Putra", "Rahardjo", "Santoso", "Setiawan", "Siregar", "Suryadi", "Wijaya", "Wibowo", "Yulianto"])
CITIES = np.array(["Jakarta", "Bandung", "Surabaya", "Medan", "Semarang", "Makassar", "Denpasar", "Yogyakarta"])
PROVINCES = np.array(["DKI Jakarta", "Jawa Barat", "Jawa Timur", "Sumatera Utara", "Jawa Tengah", "Sulawesi Selatan", "Bali", "DI Yogyakarta"])
COUNTRIES = np.array(["ID", "SG", "MY", "AU", "JP", "US", "GB", "AE"])
FX = {"IDR": 1.0, "USD": 16_200.0, "SGD": 12_050.0, "EUR": 17_600.0}

def names(n, international=False):
    first_pool = np.array(["Alexander", "Dmitri", "Elena", "Farid", "Hassan", "Ivan", "Karim", "Laila", "Omar", "Sergei"]) if international else FIRST
    last_pool = np.array(["Petrov", "Ivanov", "Karimov", "Volkov", "Haddad", "Bashir", "Abramov", "Sokolov"]) if international else LAST
    first, last = rng.choice(first_pool, n), rng.choice(last_pool, n)
    return first, last, np.char.add(np.char.add(first, " "), last)

def recent_date(days_back):
    return (pd.Timestamp("2026-06-30") - pd.to_timedelta(days_back, unit="D")).date()

pd.Series(cfg, name="configured_value").to_frame()


,configured_value
customers,10000
accounts,15000
counterparties,5000
watchlist,1200
transactions,250000
history_days,240
labels_per_scenario,100
sanctions_cases,750


## 3. Generate customers, accounts, counterparties, and sanctions watchlist

Every identity is synthetic. These master tables establish the profile and relationship context for transactions.

In [3]:
n = cfg["customers"]
first, last, full_name = names(n)
city_ix = rng.integers(0, len(CITIES), n)
customer_country = rng.choice(COUNTRIES, n, p=[0.88, 0.03, 0.025, 0.015, 0.015, 0.015, 0.01, 0.01])
income = np.clip(rng.lognormal(np.log(12_000_000), 0.72, n), 3_000_000, 450_000_000).round(-3)

customers = pd.DataFrame({
    "customer_id": [f"CUS{i:07d}" for i in range(1, n + 1)], "full_name": full_name, "first_name": first,
    "middle_name": rng.choice(["", "A.", "M.", "R."], n, p=[0.76, 0.08, 0.08, 0.08]), "last_name": last,
    "date_of_birth": [recent_date(int(value)) for value in rng.integers(8_000, 27_000, n)], "gender": rng.choice(["Female", "Male"], n),
    "nationality": np.where(customer_country == "ID", "Indonesian", customer_country), "id_type": rng.choice(["KTP", "Passport", "Other"], n, p=[0.90, 0.08, 0.02]),
    "id_number": [f"SYN{value:012d}" for value in rng.integers(1, 10**12, n)], "address_line_1": [f"Jl. {LAST[i % len(LAST)]} No. {rng.integers(1, 220)}" for i in range(n)],
    "address_line_2": rng.choice(["", "Tower A", "Unit 12", "Blok B"], n), "city": CITIES[city_ix], "province": PROVINCES[city_ix],
    "postal_code": rng.integers(10110, 99990, n).astype(str), "country": customer_country,
    "phone_number": [f"+628{value:09d}" for value in rng.integers(10**8, 10**9, n)], "email": [f"customer{i:07d}@synthetic.vteki.test" for i in range(1, n + 1)],
    "occupation": rng.choice(["Teacher", "Engineer", "Entrepreneur", "Accountant", "Doctor", "Consultant", "Trader", "Civil Servant"], n),
    "employer_name": [f"PT {LAST[i % len(LAST)]} Synthetic" for i in range(n)], "monthly_income": income,
    "customer_segment": rng.choice(["Retail", "SME", "Corporate", "Priority"], n, p=[0.68, 0.18, 0.06, 0.08]),
    "customer_risk_rating": rng.choice(["Low", "Medium", "High"], n, p=[0.70, 0.24, 0.06]), "pep_flag": rng.choice([0, 1], n, p=[0.985, 0.015]),
    "onboarding_date": [recent_date(int(value)) for value in rng.integers(20, 2_400, n)], "account_status": rng.choice(["Active", "Dormant", "Closed"], n, p=[0.935, 0.055, 0.01]),
})

owners = np.concatenate([np.arange(n), rng.integers(0, n, cfg["accounts"] - n)])
rng.shuffle(owners)
accounts = pd.DataFrame({
    "account_id": [f"ACC{i:08d}" for i in range(1, cfg["accounts"] + 1)], "customer_id": customers.iloc[owners]["customer_id"].to_numpy(),
    "account_number": [f"88{value:011d}" for value in rng.integers(1, 10**11, cfg["accounts"])], "account_type": rng.choice(["Saving", "Current", "Business"], cfg["accounts"], p=[0.68, 0.20, 0.12]),
    "currency": rng.choice(["IDR", "USD", "SGD", "EUR"], cfg["accounts"], p=[0.87, 0.07, 0.04, 0.02]), "branch_code": [f"BR{value:04d}" for value in rng.integers(1, 350, cfg["accounts"])],
    "opening_date": [recent_date(int(value)) for value in rng.integers(20, 2_500, cfg["accounts"])], "account_status": rng.choice(["Active", "Dormant", "Closed"], cfg["accounts"], p=[0.94, 0.05, 0.01]),
    "average_balance": np.clip(customers.iloc[owners]["monthly_income"].to_numpy() * rng.lognormal(0.2, 0.8, cfg["accounts"]), 100_000, 8_000_000_000).round(2),
    "risk_level": rng.choice(["Low", "Medium", "High"], cfg["accounts"], p=[0.72, 0.23, 0.05]),
})
accounts["current_balance"] = (accounts["average_balance"] * rng.uniform(0.25, 1.85, len(accounts))).round(2)

cp_first, cp_last, cp_person = names(cfg["counterparties"])
cp_type = rng.choice(["Individual", "Company"], cfg["counterparties"], p=[0.72, 0.28])
cp_country = rng.choice(COUNTRIES, cfg["counterparties"], p=[0.64, 0.09, 0.08, 0.04, 0.04, 0.04, 0.035, 0.035])
counterparties = pd.DataFrame({
    "counterparty_id": [f"CP{i:07d}" for i in range(1, cfg["counterparties"] + 1)], "counterparty_name": np.where(cp_type == "Individual", cp_person, [f"PT {value} Synthetic" for value in cp_last]),
    "entity_type": cp_type, "alias_name": np.char.add(cp_first, np.char.add(" ", cp_last)), "date_of_birth": np.where(cp_type == "Individual", "1985-01-01", ""),
    "registration_date": np.where(cp_type == "Company", "2018-01-01", ""), "address": [f"{value} Synthetic Avenue" for value in rng.integers(1, 900, cfg["counterparties"])],
    "city": rng.choice(CITIES, cfg["counterparties"]), "country": cp_country, "nationality": cp_country, "bank_name": rng.choice(["Bank Nusantara", "Bank Sentra", "Asia Commerce Bank"], cfg["counterparties"]),
    "account_number": [f"77{value:010d}" for value in rng.integers(1, 10**10, cfg["counterparties"])], "industry": rng.choice(["Retail", "Logistics", "Technology", "Construction", "Healthcare", "Manufacturing"], cfg["counterparties"]),
    "risk_level": rng.choice(["Low", "Medium", "High"], cfg["counterparties"], p=[0.65, 0.28, 0.07]),
})

wl_first, wl_last, wl_name = names(cfg["watchlist"], international=True)
wl_type = rng.choice(["Individual", "Organization"], cfg["watchlist"], p=[0.78, 0.22])
wl_primary = np.where(wl_type == "Individual", wl_name, np.char.add(wl_last, " Global Holdings"))
watchlist = pd.DataFrame({
    "watchlist_id": [f"WL{i:06d}" for i in range(1, cfg["watchlist"] + 1)], "entity_type": wl_type, "primary_name": wl_primary,
    "first_name": np.where(wl_type == "Individual", wl_first, ""), "middle_name": "", "last_name": np.where(wl_type == "Individual", wl_last, ""),
    "aliases": np.where(wl_type == "Individual", np.char.add(wl_first, np.char.add(" ", wl_last)), np.char.add(wl_last, " Group")), "date_of_birth": np.where(wl_type == "Individual", "1978-04-12", ""),
    "place_of_birth": rng.choice(["Synthetic City A", "Synthetic City B", "Unknown"], cfg["watchlist"]), "nationality": rng.choice(["RU", "SY", "KP"], cfg["watchlist"]),
    "address": [f"{value} Watchlist Street" for value in rng.integers(1, 400, cfg["watchlist"])], "city": rng.choice(["Synthetic City A", "Synthetic City B"], cfg["watchlist"]),
    "country": rng.choice(["RU", "SY", "KP"], cfg["watchlist"]), "passport_number": [f"SYN-P-{value:08d}" for value in rng.integers(1, 10**8, cfg["watchlist"])],
    "national_id": [f"SYN-N-{value:09d}" for value in rng.integers(1, 10**9, cfg["watchlist"])], "organization_name": np.where(wl_type == "Organization", wl_primary, ""),
    "program": rng.choice(["Synthetic Global", "Synthetic Regional", "Internal Watchlist"], cfg["watchlist"]), "listing_reason": "Synthetic prototype record", "list_source": "Synthetic Prototype",
    "listing_date": "2024-01-01", "active_flag": rng.choice([1, 0], cfg["watchlist"], p=[0.96, 0.04]), "risk_level": rng.choice(["High", "Critical"], cfg["watchlist"]),
})

pd.DataFrame({"table": ["customers", "accounts", "counterparties", "sanctions_watchlist"], "rows": [len(customers), len(accounts), len(counterparties), len(watchlist)]})


,table,rows
0,customers,10000
1,accounts,15000
2,counterparties,5000
3,sanctions_watchlist,1200


## 4. Generate baseline transaction data

One row represents one transaction. The schema includes sender, receiver, beneficiary, amount, country, device, channel, purpose, and status fields required by the PDF.

In [4]:
n_tx = cfg["transactions"]
active_accounts = accounts.loc[accounts["account_status"] != "Closed"].reset_index(drop=True)
sender = active_accounts.iloc[rng.integers(0, len(active_accounts), n_tx)].reset_index(drop=True)
receiver = active_accounts.iloc[rng.integers(0, len(active_accounts), n_tx)].reset_index(drop=True)
sender_customer = customers.set_index("customer_id").loc[sender["customer_id"]].reset_index()
receiver_customer = customers.set_index("customer_id").loc[receiver["customer_id"]].reset_index()
cp = counterparties.iloc[rng.integers(0, len(counterparties), n_tx)].reset_index(drop=True)
internal = rng.random(n_tx) < 0.28
seconds = np.sort(rng.integers(0, cfg["history_days"] * 86_400, n_tx))
timestamps = pd.Timestamp("2026-06-30 23:59:59") - pd.to_timedelta(cfg["history_days"] * 86_400 - seconds, unit="s")
amount_idr = np.clip(rng.lognormal(np.log(2_800_000), 1.15, n_tx), 25_000, 350_000_000).round(2)
currency = rng.choice(["IDR", "USD", "SGD", "EUR"], n_tx, p=[0.88, 0.07, 0.03, 0.02])
fx_rate = pd.Series(currency).map(FX).to_numpy(float)

transactions = pd.DataFrame({
    "transaction_id": [f"TXN{i:010d}" for i in range(1, n_tx + 1)], "transaction_timestamp": timestamps,
    "sender_customer_id": sender["customer_id"].to_numpy(), "sender_account_id": sender["account_id"].to_numpy(), "sender_name": sender_customer["full_name"].to_numpy(), "sender_address": sender_customer["address_line_1"].to_numpy(), "sender_country": sender_customer["country"].to_numpy(),
    "receiver_customer_id": np.where(internal, receiver["customer_id"], ""), "receiver_account_id": np.where(internal, receiver["account_id"], ""), "receiver_name": np.where(internal, receiver_customer["full_name"], cp["counterparty_name"]),
    "receiver_address": np.where(internal, receiver_customer["address_line_1"], cp["address"]), "receiver_country": np.where(internal, receiver_customer["country"], cp["country"]),
    "beneficiary_name": np.where(internal, receiver_customer["full_name"], cp["counterparty_name"]), "beneficiary_address": np.where(internal, receiver_customer["address_line_1"], cp["address"]), "counterparty_id": np.where(internal, "", cp["counterparty_id"]),
    "transaction_type": rng.choice(["Transfer", "Cash", "RTGS", "SWIFT", "BI-FAST"], n_tx, p=[0.34, 0.12, 0.09, 0.10, 0.35]), "channel": rng.choice(["Mobile", "Internet", "Branch", "ATM", "API"], n_tx, p=[0.42, 0.25, 0.09, 0.14, 0.10]),
    "amount": (amount_idr / fx_rate).round(2), "currency": currency, "amount_idr_equivalent": amount_idr, "debit_credit": "Debit", "purpose_code": rng.choice(["SALARY", "FAMILY", "TRADE", "BILL", "INVESTMENT", "OTHER"], n_tx),
    "purpose_description": rng.choice(["Monthly payment", "Family support", "Invoice settlement", "Goods purchase", "Investment transfer", "Other transfer"], n_tx), "reference_number": [f"REF{value:011d}" for value in rng.integers(1, 10**11, n_tx)],
    "source_of_fund": rng.choice(["Salary", "Business", "Investment", "Unknown"], n_tx, p=[0.48, 0.30, 0.12, 0.10]), "destination_bank": rng.choice(["Bank Nusantara", "Bank Sentra", "Asia Commerce Bank"], n_tx),
    "destination_country": np.where(internal, receiver_customer["country"], cp["country"]), "ip_address": [f"10.{a}.{b}.{c}" for a, b, c in rng.integers(1, 255, (n_tx, 3))], "device_id": [f"DEV{value:08d}" for value in rng.integers(1, max(2, cfg["customers"] * 2), n_tx)],
    "latitude": rng.normal(-6.2, 2.4, n_tx).round(6), "longitude": rng.normal(106.8, 5.1, n_tx).round(6), "transaction_status": rng.choice(["Success", "Failed", "Reversed"], n_tx, p=[0.975, 0.018, 0.007]),
})
transactions.head(3)


,transaction_id,transaction_timestamp,sender_customer_id,sender_account_id,sender_name,sender_address,sender_country,receiver_customer_id,receiver_account_id,receiver_name,receiver_address,receiver_country,beneficiary_name,beneficiary_address,counterparty_id,...,amount,currency,amount_idr_equivalent,debit_credit,purpose_code,purpose_description,reference_number,source_of_fund,destination_bank,destination_country,ip_address,device_id,latitude,longitude,transaction_status
0,TXN0000000001,2025-11-03 00:00:18,CUS0004481,ACC00003723,Hana Wibowo,Jl. Rahardjo No. 213,ID,CUS0007767,ACC00003390,Rizky Chandra,Jl. Prakoso No. 101,ID,Rizky Chandra,Jl. Prakoso No. 101,,...,"6,211,901.22",IDR,"6,211,901.22",Debit,OTHER,Monthly payment,REF77006270071,Salary,Bank Sentra,ID,10.224.13.182,DEV00010054,-3.53,110.08,Success
1,TXN0000000002,2025-11-03 00:02:26,CUS0007453,ACC00007015,Rani Suryadi,Jl. Nugraha No. 22,ID,,,Gita Adinata,101 Synthetic Avenue,ID,Gita Adinata,101 Synthetic Avenue,CP0004049,...,"1,347,876.32",IDR,"1,347,876.32",Debit,INVESTMENT,Investment transfer,REF29442744933,Business,Bank Sentra,ID,10.155.45.84,DEV00018311,-8.28,108.08,Success
2,TXN0000000003,2025-11-03 00:03:18,CUS0005894,ACC00000669,Dimas Wibowo,Jl. Permana No. 198,ID,,,Raka Santoso,92 Synthetic Avenue,ID,Raka Santoso,92 Synthetic Avenue,CP0003095,...,"987,476.37",IDR,"987,476.37",Debit,BILL,Investment transfer,REF48061658365,Business,Asia Commerce Bank,ID,10.108.231.12,DEV00013796,-6.24,110.91,Success


## 5. Inject the ten AML scenarios and sanctions-name variations

Ground truth is kept in separate files. It is for evaluation later and must never become an input feature.

In [5]:
SCENARIOS = [
    ("AML-S01", "Structuring / Smurfing", "High"), ("AML-S02", "Sudden Transaction Spike", "High"), ("AML-S03", "Rapid Movement of Funds", "Critical"),
    ("AML-S04", "Dormant Account Reactivation", "Critical"), ("AML-S05", "Multiple Senders to One Receiver", "High"), ("AML-S06", "One Sender to Multiple Beneficiaries", "High"),
    ("AML-S07", "Circular Transaction", "Critical"), ("AML-S08", "High-Risk Geography", "High"), ("AML-S09", "Unusual Transaction Purpose", "High"), ("AML-S10", "Potential Mule Account", "Critical"),
]
labelled_rows = len(SCENARIOS) * cfg["labels_per_scenario"]
selected = rng.choice(transactions.index.to_numpy(), labelled_rows, replace=False)
truth_rows = []

for scenario_number, (scenario_id, scenario_name, expected_risk) in enumerate(SCENARIOS):
    idx = selected[scenario_number * cfg["labels_per_scenario"]:(scenario_number + 1) * cfg["labels_per_scenario"]]
    transactions.loc[idx, "transaction_status"] = "Success"
    group_id = [f"{scenario_id}-GRP-{i // 4 + 1:04d}" for i in range(len(idx))]
    if scenario_id == "AML-S01":
        values = rng.uniform(9_100_000, 9_850_000, len(idx)).round(2)
        transactions.loc[idx, "amount"] = values; transactions.loc[idx, "amount_idr_equivalent"] = values; transactions.loc[idx, "currency"] = "IDR"
    elif scenario_id == "AML-S02":
        values = rng.uniform(150_000_000, 520_000_000, len(idx)).round(2)
        transactions.loc[idx, "amount"] = values; transactions.loc[idx, "amount_idr_equivalent"] = values; transactions.loc[idx, "currency"] = "IDR"
    elif scenario_id == "AML-S03":
        transactions.loc[idx, "purpose_description"] = "Synthetic rapid pass-through movement"
        transactions.loc[idx, "amount_idr_equivalent"] = rng.uniform(350_000_000, 500_000_000, len(idx))
    elif scenario_id == "AML-S04":
        dormant = accounts.loc[accounts["account_status"] == "Dormant", "account_id"].to_numpy()
        transactions.loc[idx, "sender_account_id"] = rng.choice(dormant, len(idx)); transactions.loc[idx, "amount_idr_equivalent"] = rng.uniform(180_000_000, 1_100_000_000, len(idx))
    elif scenario_id == "AML-S05":
        receiver_name = "Synthetic Funnel Beneficiary"; transactions.loc[idx, ["receiver_name", "beneficiary_name"]] = receiver_name
    elif scenario_id == "AML-S06":
        transactions.loc[idx, "sender_account_id"] = transactions.loc[idx[0], "sender_account_id"]
        transactions.loc[idx, "beneficiary_name"] = [f"Synthetic Beneficiary {i:04d}" for i in range(len(idx))]
    elif scenario_id == "AML-S07":
        transactions.loc[idx, "receiver_customer_id"] = transactions.loc[idx, "sender_customer_id"].shift(-1, fill_value=transactions.loc[idx[0], "sender_customer_id"])
    elif scenario_id == "AML-S08":
        countries = rng.choice(["RU", "SY", "KP"], len(idx))
        transactions.loc[idx, "receiver_country"] = countries; transactions.loc[idx, "destination_country"] = countries
    elif scenario_id == "AML-S09":
        values = rng.uniform(350_000_000, 900_000_000, len(idx)).round(2)
        transactions.loc[idx, "purpose_code"] = "INDUSTRIAL"; transactions.loc[idx, "purpose_description"] = "Industrial machinery procurement"
        transactions.loc[idx, "amount"] = values; transactions.loc[idx, "amount_idr_equivalent"] = values; transactions.loc[idx, "currency"] = "IDR"
    elif scenario_id == "AML-S10":
        transactions.loc[idx, "purpose_description"] = "Synthetic mule-account movement"; transactions.loc[idx, "amount_idr_equivalent"] = rng.uniform(35_000_000, 180_000_000, len(idx))
    for position, row_idx in enumerate(idx):
        truth_rows.append({"transaction_id": transactions.loc[row_idx, "transaction_id"], "customer_id": transactions.loc[row_idx, "sender_customer_id"], "scenario_id": scenario_id, "scenario_name": scenario_name, "injected_flag": 1, "expected_risk": expected_risk, "notes": f"Synthetic injection for {scenario_name}.", "scenario_group_id": group_id[position]})

aml_ground_truth = pd.DataFrame(truth_rows)

active_watchlist = watchlist.loc[watchlist["active_flag"] == 1].reset_index(drop=True)
sanction_indices = rng.choice(transactions.index.difference(selected), cfg["sanctions_cases"], replace=False)
variations = ["Exact", "Alias", "Reversed", "Abbreviation", "Spacing", "Typo"]
sanctions_rows = []
for row_idx in sanction_indices:
    watch = active_watchlist.iloc[rng.integers(0, len(active_watchlist))]
    variation = str(rng.choice(variations)); tokens = str(watch["primary_name"]).split()
    if variation == "Alias": injected = str(watch["aliases"])
    elif variation == "Reversed": injected = " ".join(reversed(tokens))
    elif variation == "Abbreviation": injected = f"{tokens[0][0]}. {' '.join(tokens[1:])}"
    elif variation == "Spacing": injected = "  ".join(tokens)
    elif variation == "Typo" and len(tokens[0]) > 3: injected = tokens[0][:-1] + "x " + " ".join(tokens[1:])
    else: injected = str(watch["primary_name"])
    transactions.loc[row_idx, ["receiver_name", "beneficiary_name", "receiver_address", "beneficiary_address", "receiver_country", "destination_country"]] = [injected, injected, watch["address"], watch["address"], watch["country"], watch["country"]]
    sanctions_rows.append({"transaction_id": transactions.loc[row_idx, "transaction_id"], "watchlist_id": watch["watchlist_id"], "injected_name": injected, "original_name": watch["primary_name"], "variation_type": variation, "expected_match": 1})

sanctions_ground_truth = pd.DataFrame(sanctions_rows)
transactions = transactions.sort_values(["transaction_timestamp", "transaction_id"]).reset_index(drop=True)
pd.DataFrame({"AML labelled transaction-scenario rows": [len(aml_ground_truth)], "Sanctions candidate cases": [len(sanctions_ground_truth)]})


,AML labelled transaction-scenario rows,Sanctions candidate cases
0,1000,750


## 6. Validate and write CSV files

Validation checks keys, foreign-key relationships, positive amounts, exactly ten AML scenarios, and traceability. Files are written only after all checks pass.

## 6. Standardize optional master-data fields and preserve transaction relationships

Some master-data fields are naturally not applicable: an organization has no date of birth and an individual has no company registration date. The generator writes documented values and indicator columns for these master-data cases.

For `transactions.csv`, the PDF explicitly states that `receiver_customer_id` and `receiver_account_id` exist **if internal**. Therefore the raw transaction table preserves this relationship: an external transfer has a real `counterparty_id` (`CP...`) and blank internal receiver IDs; an internal transfer has real receiver customer/account IDs (`CUS...` / `ACC...`) and a blank `counterparty_id`. No artificial external/internal ID label is exported. These structural blanks are handled later in preprocessing and feature engineering.

In [ ]:
# Re-align transaction party attributes to the account master after scenario injection.
# This keeps every sender/receiver account relationship valid before CSV export.
account_customer_lookup = accounts.set_index("account_id")["customer_id"]
customer_lookup_for_repair = customers.set_index("customer_id")

sender_customer_ids = account_customer_lookup.reindex(transactions["sender_account_id"]).to_numpy()
sender_profile = customer_lookup_for_repair.reindex(sender_customer_ids)
transactions["sender_customer_id"] = sender_customer_ids
transactions["sender_name"] = sender_profile["full_name"].to_numpy()
transactions["sender_address"] = sender_profile["address_line_1"].to_numpy()
transactions["sender_country"] = sender_profile["country"].to_numpy()

external_receiver_mask = transactions["receiver_account_id"].eq("")
transactions.loc[external_receiver_mask, "receiver_customer_id"] = ""

internal_receiver_mask = transactions["receiver_account_id"].ne("")
receiver_customer_ids = account_customer_lookup.reindex(transactions.loc[internal_receiver_mask, "receiver_account_id"]).to_numpy()
receiver_profile = customer_lookup_for_repair.reindex(receiver_customer_ids)
transactions.loc[internal_receiver_mask, "receiver_customer_id"] = receiver_customer_ids
transactions.loc[internal_receiver_mask, "receiver_name"] = receiver_profile["full_name"].to_numpy()
transactions.loc[internal_receiver_mask, "receiver_address"] = receiver_profile["address_line_1"].to_numpy()
transactions.loc[internal_receiver_mask, "receiver_country"] = receiver_profile["country"].to_numpy()
transactions.loc[internal_receiver_mask, "beneficiary_name"] = receiver_profile["full_name"].to_numpy()
transactions.loc[internal_receiver_mask, "beneficiary_address"] = receiver_profile["address_line_1"].to_numpy()
transactions.loc[internal_receiver_mask, "destination_country"] = receiver_profile["country"].to_numpy()

# Use explicit values rather than blank strings. This prevents pandas from reading blanks as NaN.
MISSING_TOKENS = {
    "no_middle_name": "NO_MIDDLE_NAME",
    "no_address_line_2": "NO_ADDRESS_LINE_2",
    "not_applicable_individual": "NOT_APPLICABLE_INDIVIDUAL",
    "not_applicable_organization": "NOT_APPLICABLE_ORGANIZATION",
    "external_customer": "EXTERNAL_NOT_BANK_CUSTOMER",
    "external_account": "EXTERNAL_ACCOUNT_NOT_ON_US",
    "internal_counterparty": "INTERNAL_ON_US_TRANSFER",
}

customers["middle_name"] = customers["middle_name"].replace("", MISSING_TOKENS["no_middle_name"]).fillna(MISSING_TOKENS["no_middle_name"])
customers["address_line_2"] = customers["address_line_2"].replace("", MISSING_TOKENS["no_address_line_2"]).fillna(MISSING_TOKENS["no_address_line_2"])

# Valid sentinel dates keep the columns parseable as dates. The indicator preserves their actual applicability.
counterparties["has_date_of_birth"] = (counterparties["entity_type"] == "Individual").astype(int)
counterparties["has_registration_date"] = (counterparties["entity_type"] == "Company").astype(int)
counterparties["date_of_birth"] = counterparties["date_of_birth"].replace("", "1900-01-01").fillna("1900-01-01")
counterparties["registration_date"] = counterparties["registration_date"].replace("", "1900-01-01").fillna("1900-01-01")

watchlist["has_date_of_birth"] = (watchlist["entity_type"] == "Individual").astype(int)
watchlist["first_name"] = watchlist["first_name"].replace("", MISSING_TOKENS["not_applicable_organization"]).fillna(MISSING_TOKENS["not_applicable_organization"])
watchlist["middle_name"] = watchlist["middle_name"].replace("", MISSING_TOKENS["no_middle_name"]).fillna(MISSING_TOKENS["no_middle_name"])
watchlist["last_name"] = watchlist["last_name"].replace("", MISSING_TOKENS["not_applicable_organization"]).fillna(MISSING_TOKENS["not_applicable_organization"])
watchlist["date_of_birth"] = watchlist["date_of_birth"].replace("", "1900-01-01").fillna("1900-01-01")
watchlist["organization_name"] = watchlist["organization_name"].replace("", MISSING_TOKENS["not_applicable_individual"]).fillna(MISSING_TOKENS["not_applicable_individual"])

# Represent the transaction relationship explicitly so the exported CSV contains no NaN values.
# These labels are synthetic placeholders, not real customer, account, or counterparty IDs.
# The raw transaction schema stays limited to the PDF fields; no derived helper columns are exported.
transactions["receiver_customer_id"] = transactions["receiver_customer_id"].replace("", "EXTERNAL_NOT_BANK_CUSTOMER").fillna("EXTERNAL_NOT_BANK_CUSTOMER")
transactions["receiver_account_id"] = transactions["receiver_account_id"].replace("", "EXTERNAL_ACCOUNT_NOT_ON_US").fillna("EXTERNAL_ACCOUNT_NOT_ON_US")
transactions["counterparty_id"] = transactions["counterparty_id"].replace("", "INTERNAL_ON_US_TRANSFER").fillna("INTERNAL_ON_US_TRANSFER")

exported_tables = {
    "customers": customers,
    "accounts": accounts,
    "counterparties": counterparties,
    "sanctions_watchlist": watchlist,
    "transactions": transactions,
    "aml_ground_truth": aml_ground_truth,
    "sanctions_ground_truth": sanctions_ground_truth,
}
missing_value_audit = pd.DataFrame(
    {
        "rows": {name: len(frame) for name, frame in exported_tables.items()},
        "null_cells": {name: int(frame.isna().sum().sum()) for name, frame in exported_tables.items()},
        "blank_text_cells": {
            name: int(frame.select_dtypes(include="object").eq("").sum().sum())
            for name, frame in exported_tables.items()
        },
    }
)
display(missing_value_audit)
assert (missing_value_audit["null_cells"] == 0).all(), "NaN values remain after standardization."
assert (missing_value_audit["blank_text_cells"] == 0).all(), "Blank text values remain after standardization."

In [ ]:
checks = {
    "customer_ids_unique": customers["customer_id"].is_unique,
    "account_ids_unique": accounts["account_id"].is_unique,
    "transaction_ids_unique": transactions["transaction_id"].is_unique,
    "account_customer_fk": accounts["customer_id"].isin(customers["customer_id"]).all(),
    "transaction_sender_customer_fk": transactions["sender_customer_id"].isin(customers["customer_id"]).all(),
    "positive_idr_amount": (transactions["amount_idr_equivalent"] > 0).all(),
    "no_missing_values": (missing_value_audit["null_cells"] == 0).all(),
    "no_blank_text_values": (missing_value_audit["blank_text_cells"] == 0).all(),
    "ten_aml_scenarios": aml_ground_truth["scenario_id"].nunique() == 10,
    "aml_truth_traceable": aml_ground_truth["transaction_id"].isin(transactions["transaction_id"]).all(),
    "sanctions_truth_traceable": sanctions_ground_truth["transaction_id"].isin(transactions["transaction_id"]).all(),
}
check_frame = pd.Series(checks, name="passed").to_frame()
display(check_frame)
assert check_frame["passed"].all(), "Generation validation failed."

customers.to_csv(RAW_DIR / "customers.csv", index=False)
accounts.to_csv(RAW_DIR / "accounts.csv", index=False)
transactions.to_csv(RAW_DIR / "transactions.csv", index=False)
counterparties.to_csv(RAW_DIR / "counterparties.csv", index=False)
watchlist.to_csv(RAW_DIR / "sanctions_watchlist.csv", index=False)
aml_ground_truth.to_csv(TRUTH_DIR / "aml_ground_truth.csv", index=False)
sanctions_ground_truth.to_csv(TRUTH_DIR / "sanctions_ground_truth.csv", index=False)

written = list(RAW_DIR.glob("*.csv")) + list(TRUTH_DIR.glob("*.csv"))
pd.DataFrame({"file": [path.relative_to(ROOT).as_posix() for path in written], "size_mb": [round(path.stat().st_size / 1_000_000, 2) for path in written]})


## Next step

Open `02_eda.ipynb` and run all cells to inspect the generated data. Feature engineering and AML rules intentionally come later.